# Bellwether — analytics primitives on Manifold (Prompt 4)

Validate `performance` / `specialization` / `ranking` on the Manifold data
loaded into the canonical tables. The same code runs on Polymarket via a
swapped ingester (Prompt 6) — that's the de-risking payoff.

Prereqs: `tt db init`, then `tt manifold load <user>` (or
`python -m bellwether_ingestion.manifold load-user <user>`), with `DATABASE_URL` set.

In [ ]:
from bellwether_analytics.core import (
    RankConfig,
    load_trades_df,
    performance_by_wallet,
    rank,
    specialization_by_wallet,
)

df = load_trades_df(platform="manifold")
print(f"{len(df)} trades, {df['wallet'].nunique()} wallets, "
      f"{int(df['resolved_at'].notna().sum())} resolved rows")
df.head()

In [ ]:
perf = performance_by_wallet(df)
perf.sort_values("realized_pnl", ascending=False).head(20)

In [ ]:
spec = specialization_by_wallet(df)
spec.sort_values("hhi", ascending=False).head(20)

In [ ]:
# Config-driven ranking. Loosen thresholds for a prototype dataset.
cfg = RankConfig(min_resolved_trades=1, min_win_rate=0.0, min_specialization=0.0)
ranked = rank(perf, spec, cfg)
ranked[["resolved_trade_count", "win_rate", "realized_pnl", "roi", "hhi", "top_category", "score"]].head(25)